# Voted Perceptron on Breast Cancer Wisconsin (Diagnostic)
Implementation of Freund & Schapire (1999), *Large Margin Classification Using the Perceptron Algorithm*, on the Kaggle Breast Cancer Wisconsin dataset.

**Questions** (the paper's questions, on new data)
1. Does **Vote** beat **Last** (the classic perceptron)?
2. How close is it to an SVM?
3. What does it cost in time?

**Setup**
- 569 tumors, 30 measurements of the cell nuclei per tumor. Label: malignant = +1 (212 tumors, 37%), benign = −1.
- Linear kernel, **d = 1**: $K(x, z) = (1 + x \cdot z)$. The "1 +" acts as an intercept.
- **T = 1 and T = 3** passes through the training data.
- Prediction rules from Section 5 of the paper: **Vote, Avg, Last, Rand**.
- 80/20 stratified train/test split. Features are standardized using training-set statistics only.
- Every number is over **10 train/test splits × 5 random training orders = 50 runs**.
- Reference: scikit-learn `SVC` with the same kernel (default C = 1, untuned).

In [1]:
import time
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

## 1. Load and clean the data

In [2]:
df = pd.read_csv("data.csv")
df = df.drop(columns=["id", "Unnamed: 32"])          # id is not a feature; Unnamed: 32 is an empty column
y_all = np.where(df.pop("diagnosis") == "M", 1, -1)  # malignant = +1, benign = -1
X_all = df.to_numpy(dtype=float)

print("X:", X_all.shape)
print("malignant (+1):", (y_all == 1).sum(), "  benign (-1):", (y_all == -1).sum())

X: (569, 30)
malignant (+1): 212   benign (-1): 357


## 2. Voted perceptron (Figure 1 of the paper)
**Training.** Go through the points one at a time. If $y \cdot \mathrm{sign}(v \cdot x) \le 0$ (a mistake), save the current $v$ with its count $c$, then update $v \leftarrow v + y\,x$ and start a new count. If correct, add 1 to the current count.

**Prediction rules**
| Rule | Score for a new point $x$ |
|---|---|
| Vote | $\sum_k c_k\,\mathrm{sign}(v_k \cdot x)$ |
| Avg | $\sum_k c_k\,(v_k \cdot x)$ |
| Last | $v_{\text{final}} \cdot x$ (the classic perceptron) |
| Rand | $v \cdot x$ for the version alive at a random time step |

**Implementation trick (Section 4).** Each $v_k$ is stored as the list of points it made mistakes on, so $v_k \cdot x = \sum_j y_j K(x_j, x)$. One cumulative sum over the mistake points gives every $v_k \cdot x$ at once, so voting costs the same as using only the last vector.

In [3]:
def kernel(A, B, d=1):
    return (1.0 + A @ B.T) ** d


class VotedPerceptron:
    def __init__(self, d=1, epochs=1, seed=0):
        self.d, self.epochs, self.seed = d, epochs, seed

    def fit(self, X, y, K=None):
        m = len(y)
        if K is None:
            K = kernel(X, X, self.d)                    # Gram matrix: K[i, j] = K(x_i, x_j)
        order = np.random.default_rng(self.seed).permutation(m)
        alpha = np.zeros(m)                             # alpha[i] = signed times x_i was added, so K[i] @ alpha = v . x_i
        mist_idx, mist_y, counts = [], [], [0]
        self.n_steps = int(round(self.epochs * m))
        for t in range(self.n_steps):
            i = order[t % m]
            y_hat = 1 if K[i] @ alpha > 0 else -1       # a tie (score 0) counts as a mistake
            if y_hat == y[i]:
                counts[-1] += 1                         # current version survives one more step
            else:
                alpha[i] += y[i]                        # v <- v + y x
                mist_idx.append(i); mist_y.append(y[i]); counts.append(1)
        self.X_train = X
        self.mist_idx = np.array(mist_idx, dtype=int)
        self.mist_y = np.array(mist_y, dtype=float)
        self.c = np.array(counts, dtype=float)          # lifetimes c_1 ... c_k
        return self

    def _all_scores(self, X):
        # column j holds v_{j+1} . x for every saved version (v_1 = 0)
        Kt = kernel(X, self.X_train[self.mist_idx], self.d)
        return np.hstack([np.zeros((len(X), 1)), np.cumsum(Kt * self.mist_y, axis=1)])

    def predict(self, X, method="vote", rng=None):
        S = self._all_scores(X)
        if method == "last":
            s = S[:, -1]
        elif method == "vote":
            s = np.sign(S) @ self.c
        elif method == "avg":
            s = S @ self.c
        elif method == "rand":                          # version alive at a uniformly random step
            rng = rng or np.random.default_rng(self.seed)
            r = rng.integers(0, self.n_steps + 1, size=len(X))
            j = np.minimum(np.searchsorted(np.cumsum(self.c), r, side="right"), len(self.c) - 1)
            s = S[np.arange(len(X)), j]
        return np.where(s > 0, 1, -1)

    @property
    def mistakes(self):
        return len(self.mist_idx)                       # = kernel evaluations per prediction

## 3. Run the experiment
For each of the 50 runs, record the test error, the training time, the prediction time per tumor, and the number of kernel evaluations per prediction.

- **Train (ms):** time to train on all ~455 training tumors, including building the Gram matrix. All four rules come from the same training run, so they share this number.
- **Predict (µs / tumor):** time to classify the 114 test tumors, divided by 114 (median of 5 repeats).
- **Kernel evals / prediction:** stored mistake points for the perceptron, support vectors for the SVM. Unlike the times, this does not depend on the machine.

In [4]:
D, TS = 1, [1, 3]
METHODS = {"vote": "Vote", "avg": "Avg", "last": "Last", "rand": "Rand"}
N_SPLITS, N_ORDERS, REPEAT = 10, 5, 5

def timed(fn, repeat=1):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter(); out = fn(); times.append(time.perf_counter() - t0)
    return out, float(np.median(times))

rec = []
for split in range(N_SPLITS):
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=split)
    mu, sd = Xtr.mean(0), Xtr.std(0)
    Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd                  # standardize with training statistics only
    K, t_gram = timed(lambda: kernel(Xtr, Xtr, D))
    for T in TS:
        for order in range(N_ORDERS):
            seed = 100 * split + order
            vp, t_fit = timed(lambda: VotedPerceptron(d=D, epochs=T, seed=seed).fit(Xtr, ytr, K=K))
            for m, name in METHODS.items():
                pred, t_pred = timed(lambda: vp.predict(Xte, m, rng=np.random.default_rng(seed + 1)), REPEAT)
                rec.append(dict(T=T, split=split, order=order, Method=name,
                                error=100 * np.mean(pred != yte),
                                train_ms=(t_fit + t_gram) * 1e3,
                                pred_us=t_pred / len(yte) * 1e6,
                                kevals=vp.mistakes))
    # SVM with the same kernel (1 + x.z)^d; deterministic for a given split, so one run per split
    svc, t_fit = timed(lambda: SVC(kernel="poly", degree=D, gamma=1.0, coef0=1.0).fit(Xtr, ytr))
    pred, t_pred = timed(lambda: svc.predict(Xte), REPEAT)
    rec.append(dict(T="SVM", split=split, order=0, Method="SVC (ref.)",
                    error=100 * np.mean(pred != yte), train_ms=t_fit * 1e3,
                    pred_us=t_pred / len(yte) * 1e6, kevals=int(svc.n_support_.sum())))

runs = pd.DataFrame(rec)
runs.groupby(["T", "Method"], sort=False).size().rename("number of runs").to_frame()

number of runs
T   Method                    
1   Vote                    50
    Avg                     50
    Last                    50
    Rand                    50
3   Vote                    50
    Avg                     50
    Last                    50
    Rand                    50
SVM SVC (ref.)              10

## 4. How the 95% confidence interval is calculated
Each run gives one test error. The table reports the **mean error across runs ± a 95% margin**:

$$\bar{e} \pm 1.96 \times \frac{s}{\sqrt{n}}$$

- $\bar{e}$ = mean test error over the $n$ runs
- $s$ = sample standard deviation of the run errors (with $n - 1$ in the denominator)
- $s / \sqrt{n}$ = **standard error**: how much the mean itself would vary if we repeated the whole experiment
- 1.96 = the normal-distribution value that covers the middle 95%

So the interval is a range that very likely contains the true average error. It is **not** the range of individual runs; single runs vary much more.

$n = 50$ for the perceptron rules (10 splits × 5 orders) and $n = 10$ for the SVM (one run per split), which is why the SVM's interval is wider.

Worked example: **Vote at T = 1**.

In [5]:
e = runs.query("T == 1 and Method == 'Vote'")["error"].to_numpy()

n    = len(e)
mean = e.mean()
s    = e.std(ddof=1)            # sample standard deviation
se   = s / np.sqrt(n)           # standard error of the mean
half = 1.96 * se                # 95% margin

print(f"n (runs)              = {n}")
print(f"mean error            = {mean:.2f}%")
print(f"std. dev. across runs = {s:.2f}")
print(f"standard error s/√n   = {s:.2f} / √{n} = {se:.3f}")
print(f"95% margin 1.96 × SE  = {half:.2f}")
print(f"=> {mean:.2f} ± {half:.2f}%   (interval {mean - half:.2f}% to {mean + half:.2f}%)")
print()
print("first 10 run errors (%):", np.round(e[:10], 2))
print("each test set has 114 tumors, so one mistake = 1/114 =", f"{100/114:.2f}%")

n (runs)              = 50
mean error            = 3.95%
std. dev. across runs = 1.46
standard error s/√n   = 1.46 / √50 = 0.207
95% margin 1.96 × SE  = 0.41
=> 3.95 ± 0.41%   (interval 3.54% to 4.35%)

first 10 run errors (%): [4.39 7.89 6.14 6.14 3.51 4.39 2.63 2.63 1.75 3.51]
each test set has 114 tumors, so one mistake = 1/114 = 0.88%


## 5. Results table (slide 15)
The same calculation applied to every row. Train and predict times are medians across runs, since timings have occasional slow outliers.

In [6]:
def ci(x):
    x = np.asarray(x)
    return f"{x.mean():.2f} ± {1.96 * x.std(ddof=1) / np.sqrt(len(x)):.2f}"

table = (runs.groupby(["T", "Method"], sort=False)
             .agg(**{"Error % (95% CI)":          ("error", ci),
                     "Train (ms)":                ("train_ms", lambda x: f"{x.median():.2f}"),
                     "Predict (µs / tumor)":      ("pred_us", lambda x: f"{x.median():.1f}"),
                     "Kernel evals / prediction": ("kevals", lambda x: f"{x.mean():.0f}")}))
table.index = table.index.set_levels([[f"T = {t}" if t != "SVM" else "SVM" for t in table.index.levels[0]],
                                      table.index.levels[1]])
table.to_csv("results_table.csv")
table

Error % (95% CI) Train (ms) Predict (µs / tumor)  \
T     Method                                                        
T = 1 Vote            3.95 ± 0.41       1.75                  0.6   
      Avg             3.77 ± 0.43       1.75                  0.5   
      Last            4.84 ± 0.66       1.75                  0.5   
      Rand            6.25 ± 0.59       1.75                  0.7   
T = 3 Vote            3.72 ± 0.49       3.07                  0.8   
      Avg             3.54 ± 0.44       3.07                  0.7   
      Last            4.33 ± 0.55       3.07                  0.7   
      Rand            5.12 ± 0.44       3.07                  0.9   
SVM   SVC (ref.)      3.95 ± 1.03       2.67                  1.7   

                 Kernel evals / prediction  
T     Method                                
T = 1 Vote                              27  
      Avg                               27  
      Last                              27  
      Rand                              27  
T = 3 Vote                              59  
      Avg                               59  
      Last                              59  
      Rand                              59  
SVM   SVC (ref.)                        34

Times are machine-dependent: the slide's numbers came from my laptop, so the times above may differ slightly. The error intervals and kernel evaluations are the same on any machine.

## 6. Is Vote really better than Last? (paired sign test)
Vote and Last are scored on the same split and order in every run, so compare them head to head. Count the runs where Vote had lower error (wins) and higher error (losses), drop ties, and ask how likely that split would be if each run were a coin flip.

In [7]:
for T in TS:
    p = runs[runs["T"] == T].pivot_table(index=["split", "order"], columns="Method", values="error")
    wins, ties, losses = (p.Vote < p.Last).sum(), (p.Vote == p.Last).sum(), (p.Vote > p.Last).sum()
    pval = binomtest(wins, wins + losses, 0.5).pvalue
    print(f"T = {T}: Vote beats Last in {wins} runs, ties {ties}, loses {losses}.  Sign test p = {pval:.3f}")

T = 1: Vote beats Last in 28 runs, ties 10, loses 12.  Sign test p = 0.017
T = 3: Vote beats Last in 26 runs, ties 8, loses 16.  Sign test p = 0.164
